In [7]:
import re
import psycopg2
from groq import Groq
import pandas as pd
from prompt import get_prompt


In [8]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio

In [17]:
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
query = 'indias total export in year 2018-19 across all commodities'

In [ ]:
instructions = """
there are two senarios for you

1) you are given with user query in txt, 
you task is to convert that text to sql statement keeping users asks in mind.
you will be provided with tables desctiption and column discription and also with some example quereis use that to genertat sql

2) you already developed the sql statemtn for user, but its has some error, you will be provided with error you task is to resolve the error and create a new sql query

retunr only sql othing else
"""

In [ ]:
create_resolve_sql = Agent(
        name="agents_sql",
        instructions=instructions,
        model="gpt-4o-mini"
)

In [33]:
message = get_prompt(query)

In [ ]:
with trace("SQL Agent"):
    result = await Runner.run(create_resolve_sql, message)

In [ ]:
instruction_sql_finder = 'You will be provided with a result given by other agent in text form, your job is to find the sql statemnt in it, and reurn only sql statement and dont enlcose the sql in quote or do not write a single extra world'

In [ ]:
sql_finder = Agent(
        name="agents_sql",
        instructions=instruction_sql_finder,
        model="gpt-4o-mini"
)

In [ ]:
with trace("SQL Agent"):
    sql = await Runner.run(sql_finder, sql_finderx.final_output)

In [ ]:
instructions_check_sql = 'You are sql checker, you are provided with sql statment, your job is to find if the sql query is compilable and if ran on snowflake it will run succsfully. if not you have to provide what error it will generate, final output should contain two filed first is possible which have yes or no binary answer and second is error. if yes then error should be blank. do not correct query yourself just give me two fiels possibility and error and error shoudl be in format as t it is generted by snowflake'

In [ ]:
sql_checker = Agent(
        name="agents_sql",
        instructions=instructions_check_sql,
        model="gpt-4o-mini"
)

In [95]:
with trace("SQL Agent"):
    checker = await Runner.run(sql_checker, message)